In [13]:
import pandas as pd

In [14]:
# Pandas DataFrames for each CSV file

customer = pd.read_csv('https://raw.githubusercontent.com/tvfukuda/Northwind_csv/main/customers.csv')
category = pd.read_csv('https://raw.githubusercontent.com/tvfukuda/Northwind_csv/main/categories.csv')
employee = pd.read_csv('https://raw.githubusercontent.com/tvfukuda/Northwind_csv/main/employees.csv')
order = pd.read_csv('https://raw.githubusercontent.com/tvfukuda/Northwind_csv/main/orders.csv')
orders_detail = pd.read_csv('https://raw.githubusercontent.com/tvfukuda/Northwind_csv/main/order_details.csv')
product = pd.read_csv('https://raw.githubusercontent.com/tvfukuda/Northwind_csv/main/products.csv')
shipper = pd.read_csv('https://raw.githubusercontent.com/tvfukuda/Northwind_csv/main/shippers.csv')
supplier = pd.read_csv('https://raw.githubusercontent.com/tvfukuda/Northwind_csv/main/suppliers.csv')

In [62]:
customer_revenue = (
    order.merge(customer, on='customer_id')
         .merge(orders_detail, on='order_id')
)

customer_revenue['revenue'] = customer_revenue['quantity'] * customer_revenue['unit_price']

top_10 = (
    customer_revenue
    .groupby('company_name')['revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)


# Optional: percentage of total revenue
total_revenue = customer_revenue['revenue'].sum()
top_10['pct_of_total'] = (top_10['revenue'] / total_revenue * 100).round(2)
print(top_10)
top_10.to_csv('top_10_customers.csv', index=False)

                   company_name    revenue  pct_of_total
0                    QUICK-Stop  117483.39          8.67
1            Save-a-lot Markets  115673.39          8.54
2                  Ernst Handel  113236.68          8.36
3  Hungry Owl All-Night Grocers   57317.39          4.23
4    Rattlesnake Canyon Grocery   52245.90          3.86
5                 Hanari Carnes   34101.15          2.52
6                Folk och fä HB   32555.55          2.40
7                Mère Paillarde   32203.90          2.38
8               Königlich Essen   31745.75          2.34
9                 Queen Cozinha   30226.10          2.23


In [61]:
# top 20 products by revenue
product_revenue = product.merge(orders_detail, on='product_id')
product_revenue['revenue'] = product_revenue['quantity'] * product_revenue['unit_price_y']

top_20_products = (
    product_revenue
    .groupby('product_name')['revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
print(top_20_products)
# top 30 products underperforming by revenue
product_revenue = product.merge(orders_detail, on='product_id')
product_revenue['revenue'] = product_revenue['quantity'] * product_revenue['unit_price_y']

top_30_underperforming_products = (
    product_revenue
    .groupby('product_name')['revenue']
    .sum()
    .sort_values(ascending=True)
    .head(30)
    .reset_index()
)
print(f'\n\n{top_30_underperforming_products}')

              product_name   revenue
0            Côte de Blaye  149984.2
1  Thüringer Rostbratwurst   87736.4
2     Raclette Courdavault   76296.0
3        Camembert Pierrot   50286.0
4           Tarte au sucre   49827.9
5   Gnocchi di nonna Alice   45121.2
6    Manjimup Dried Apples   44742.6
7             Alice Mutton   35482.2
8         Carnarvon Tigers   31987.5
9        Rössle Sauerkraut   26865.6


                       product_name  revenue
0                         Chocolade  1542.75
1                           Geitost  1713.50
2                      Genen Shouyu  1813.50
3         Laughing Lumberjack Lager  2562.00
4                     Longlife Tofu  2566.00
5                        Gravad lax  3047.20
6                     Aniseed Syrup  3080.00
7                          Filo Mix  3383.80
8                  Valkoinen suklaa  3510.00
9         Louisiana Hot Spiced Okra  3519.00
10          NuNuCa Nuß-Nougat-Creme  4051.60
11                       Röd Kaviar  4200.00
12    

In [ ]:
#revenue by category and product
category_products_revenue = (
    product.merge(category, on="category_id")
           .merge(orders_detail, on="product_id")
)
category_products_revenue['revenue'] = category_products_revenue['quantity'] * category_products_revenue['unit_price_y']
category_revenue = category_products_revenue.groupby(['category_name','product_name']).agg({'revenue':'sum'}).sort_values(by='revenue', ascending=False).reset_index()
print(f'\n\n{category_revenue.head(5)}')

#revenue, quantity by category and product
category_products_revenue = (
    product.merge(category, on="category_id")
           .merge(orders_detail, on="product_id")
)
category_products_revenue['revenue'] = category_products_revenue['quantity'] * category_products_revenue['unit_price_y']
category_revenue = category_products_revenue.groupby(['category_name','product_name']).agg({'quantity':'sum','revenue':'sum'}).sort_values(by='quantity', ascending=True).reset_index()
print(f'\n\n{category_revenue.head(20)}')



    category_name             product_name   revenue
0       Beverages            Côte de Blaye  149984.2
1    Meat/Poultry  Thüringer Rostbratwurst   87736.4
2  Dairy Products     Raclette Courdavault   76296.0
3  Dairy Products        Camembert Pierrot   50286.0
4     Confections           Tarte au sucre   49827.9


     category_name                  product_name  quantity   revenue
0     Meat/Poultry               Mishi Kobe Niku        95   8827.00
1       Condiments                  Genen Shouyu       122   1813.50
2          Seafood                    Gravad lax       125   3047.20
3      Confections                     Chocolade       138   1542.75
4        Beverages     Laughing Lumberjack Lager       184   2562.00
5      Confections              Valkoinen suklaa       235   3510.00
6       Condiments     Louisiana Hot Spiced Okra       239   3519.00
7          Seafood                    Röd Kaviar       293   4200.00
8   Dairy Products            Mascarpone Fabioli       29

In [ ]:
#employee performance

employee_revenue = employee.merge(order, on='employee_id').merge(orders_detail, on='order_id')
employee_revenue['revenue']=employee_revenue['quantity'] * employee_revenue['unit_price']
employee_performance= employee_revenue.groupby(['last_name','first_name']).agg({'revenue': "sum"}).sort_values(by='revenue', ascending=False).reset_index()
print(f'{employee_performance}')





   last_name first_name    revenue
0    Peacock   Margaret  250187.45
1  Leverling      Janet  213051.30
2    Davolio      Nancy  202143.71
3     Fuller     Andrew  177749.26
4       King     Robert  141295.99
5   Callahan      Laura  133301.03
6  Dodsworth       Anne   82964.00
7     Suyama    Michael   78198.10
8   Buchanan     Steven   75567.75


In [54]:
#order differences by customers

customer_order_difference =customer.merge(order, how='inner', on='customer_id').merge(orders_detail, on='order_id')
customer_order_difference['revenue']= customer_order_difference['quantity']* customer_order_difference['unit_price']

customer_order_dif= (customer_order_difference.groupby('company_name').agg(order_count=('order_id', 'nunique'), total_revenue=('revenue', 'sum')).reset_index())
customer_order_dif['avg_order_value']= (customer_order_dif['total_revenue']/customer_order_dif['order_count']).round()
customer_order_dif=customer_order_dif.sort_values('avg_order_value', ascending=False)
print(customer_order_dif)


            

                     company_name  order_count  total_revenue  avg_order_value
59                     QUICK-Stop           28      117483.39           4196.0
20                   Ernst Handel           30      113236.68           3775.0
69             Save-a-lot Markets           31      115673.39           3731.0
36   Hungry Owl All-Night Grocers           19       57317.39           3017.0
63     Rattlesnake Canyon Grocery           18       52245.90           2903.0
..                            ...          ...            ...              ...
52                    North/South            3         649.00            216.0
44           Lazy K Kountry Store            2         357.00            178.0
43  Laughing Bacchus Wine Cellars            3         522.50            174.0
29         Galería del gastrónomo            5         836.70            167.0
12     Centro comercial Moctezuma            1         100.80            101.0

[89 rows x 4 columns]


In [60]:
#monthly sales trend

sales_trend = order.merge(orders_detail, how='left', on='order_id')
sales_trend['revenue'] = sales_trend['quantity'] * sales_trend['unit_price']

# Make sure order_date is datetime
sales_trend['order_date'] = pd.to_datetime(sales_trend['order_date'])

# Group by month
sales_trend['month'] = sales_trend['order_date'].dt.to_period('M')

monthly_sales = (
    sales_trend
    .groupby('month')['revenue']
    .sum()
    .reset_index(name='total_revenue')
)

print(monthly_sales)

      month  total_revenue
0   1996-07       30192.10
1   1996-08       26609.40
2   1996-09       27636.00
3   1996-10       41203.60
4   1996-11       49704.00
5   1996-12       50953.40
6   1997-01       66692.80
7   1997-02       41207.20
8   1997-03       39979.90
9   1997-04       55699.39
10  1997-05       56823.70
11  1997-06       39088.00
12  1997-07       55464.93
13  1997-08       49981.69
14  1997-09       59733.02
15  1997-10       70328.50
16  1997-11       45913.36
17  1997-12       77476.26
18  1998-01      100854.72
19  1998-02      104561.95
20  1998-03      109825.45
21  1998-04      134630.56
22  1998-05       19898.66


In [ ]:
print(f'customer.columns: {customer.columns.to_list()}')
print(f'category.columns: {category.columns.to_list()}')
print(f'employee.columns: {employee.columns.to_list()}')
print(f'order.columns: {order.columns.to_list()}')
print(f'orders_detail.columns: {orders_detail.columns.to_list()}')
print(f'product.columns: {product.columns.to_list()}')
print(f'shipper.columns: {shipper.columns.to_list()}')
print(f'supplier.columns: {supplier.columns.to_list()}')

customer.columns: ['customer_id', 'company_name', 'contact_name', 'contact_title', 'address', 'city', 'region', 'postal_code', 'country', 'phone', 'fax']
category.columns: ['category_id', 'category_name', 'description', 'picture']
employee.columns: ['employee_id', 'last_name', 'first_name', 'title', 'title_of_courtesy', 'birth_date', 'hire_date', 'address', 'city', 'region', 'postal_code', 'country', 'home_phone', 'extension', 'photo', 'notes', 'reports_to', 'photo_path']
order.columns: ['order_id', 'customer_id', 'employee_id', 'order_date', 'required_date', 'shipped_date', 'ship_via', 'freight', 'ship_name', 'ship_address', 'ship_city', 'ship_region', 'ship_postal_code', 'ship_country']
orders_detail.columns: ['order_id', 'product_id', 'unit_price', 'quantity', 'discount']
product.columns: ['product_id', 'product_name', 'supplier_id', 'category_id', 'quantity_per_unit', 'unit_price', 'units_in_stock', 'units_on_order', 'reorder_level', 'discontinued']
shipper.columns: ['shipper_id', 